In [ ]:
!pip install transformers torch datasets
!pip install transformers datasets torch scikit-learn
!pip install torch torchvision transformers timm pillow
!git clone https://github.com/haotian-liu/LLaVA.git
!pip install transformers torch torchvision
%cd LLaVA

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
print(f"Max position embeddings: {model.config.max_position_embeddings}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Max position embeddings: 514


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer
label_mapping = {"positive": 0, "Neutral": 1, "Negative": 2}
# Load data
train_df = pd.read_csv('../dataset/testing/test_data_post.csv')
val_df = pd.read_csv('../dataset/validation/validation_data_post.csv')
test_df = pd.read_csv('../dataset/testing/test_data_post.csv')

train_df = train_df.rename(columns={'post_mood_PNN_text': 'labels'})
val_df = val_df.rename(columns={'post_mood_PNN_text': 'labels'})
test_df = test_df.rename(columns={'post_mood_PNN_text': 'labels'})

train_df["labels"] = train_df["labels"].map(label_mapping)
val_df["labels"] = val_df["labels"].map(label_mapping)
test_df["labels"] = test_df["labels"].map(label_mapping)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

# Tokenize data
def tokenize_function(examples):
    examples["body"] = [str(text) if text else "" for text in examples["body"]]  # Ensure valid input or empty strings.
    return tokenizer(examples["body"], padding="max_length", truncation=True, max_length=512)

train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch




# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    num_labels=3  # Assuming you have three sentiment classes: positive, neutral, negative
)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="run1",
    evaluation_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
)

# Compute metrics
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-4-c7cc9d90db7a>:40: FutureWarning: `tokeniz

In [ ]:
import os
import wandb
os.environ["WANDB_DISABLED"] = "true"
wandb.init(
    project="your_project_name",  # Replace with your project name
    name="run1",  # Optional: name of the run
    config=training_args.to_dict()  # Log training arguments
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rachanavenati (rachanavenati-bauhaus-universit-t-weimar) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:

trainer.train()
trainer.save_model("../models/roberta_finetuned_sentiment")
torch.save(model.state_dict(), '../models/roberta_finetuned.pth')



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.081682,0.427419,0.319103
2,No log,0.663887,0.637097,0.634974
3,No log,0.597863,0.750000,0.750487
4,No log,0.835316,0.733871,0.728124
5,No log,0.775546,0.774194,0.774406
6,No log,0.811687,0.774194,0.774406


In [ ]:
results = trainer.evaluate(test_dataset)
label_mapping = {0: "positive", 1: "Neutral", 2: "Negative"}
print("Test Results:", results)
predictions = trainer.predict(test_dataset)
predicted_labels = predictions.predictions.argmax(axis=-1)

predicted_labels_str = [label_mapping[label] for label in predicted_labels]

test_df = pd.read_csv('../dataset/testing/test_data_post.csv')

# Add the predicted labels as a new column
test_df['predicted_label'] = predicted_labels_str

# Add the evaluation results as new columns in the DataFrame
test_df['eval_loss'] = results['eval_loss']
test_df['eval_accuracy'] = results['eval_accuracy']
test_df['eval_f1'] = results['eval_f1']
test_df['eval_runtime'] = results['eval_runtime']
test_df['eval_samples_per_second'] = results['eval_samples_per_second']
test_df['eval_steps_per_second'] = results['eval_steps_per_second']
test_df['epoch'] = results['epoch']

# Save the updated DataFrame to a new CSV file
test_df.to_csv('../outputs/updated_test_data_with_predictions.csv', index=False)


Test Results: {'eval_loss': 0.008756568655371666, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_runtime': 3.2321, 'eval_samples_per_second': 38.056, 'eval_steps_per_second': 4.95, 'epoch': 6.0}


In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Load the CSV file into a DataFrame
df = pd.read_csv('../outputs/updated_test_data_with_predictions.csv')

# Extract predictions and ground truth labels
predictions = df['predicted_label']
ground_truth = df['post_mood_PNN_text']

# Calculate precision, recall, f1-score, and accuracy for multi-class classification
precision = precision_score(ground_truth, predictions, average='weighted')  # or 'micro' or 'macro'
recall = recall_score(ground_truth, predictions, average='weighted')  # or 'micro' or 'macro'
f1 = f1_score(ground_truth, predictions, average='weighted')  # or 'micro' or 'macro'
accuracy = accuracy_score(ground_truth, predictions)

# Print the results
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')
print(f'Accuracy: {accuracy}')


Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Accuracy: 1.0


In [ ]:
###baseline
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load tokenizer and model
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Load input CSV file
input_file = "../dataset/testing/test_data_post.csv"  # Replace with your file name
output_file = "../outputs/predictions_baseline.csv"  # File to save the results

# Read the CSV file
data = pd.read_csv(input_file)

# Ensure the column with text data is correctly identified
text_column = "text"  # Replace with your column name containing the text

# Define a mapping for class labels
label_mapping = {
    0: "Negative",  # Assuming class 0 is Negative
    1: "Neutral",   # Assuming class 1 is Neutral
    2: "positive"   # Assuming class 2 is Positive
}

# Function to make predictions
def predict_sentiment(text):
    try:
        # Tokenize and truncate input
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=model.config.max_position_embeddings, padding=True)

        # Perform inference
        with torch.no_grad():
            outputs = model(**inputs)

        # Get probabilities and predicted class
        predictions = torch.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(predictions).item()
        return label_mapping[predicted_class]  # Map numerical class to label
    except Exception as e:
        return "Error"  # Handle any errors gracefully

# Apply the prediction function to the dataset
data["predicted_sentiment"] = data["body"].apply(predict_sentiment)

# Save the updated dataset to a new CSV file
data.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Predictions saved to ../outputs/predictions_baseline.csv


LLAVA MODEL FOR IMAGES ONLY

In [ ]:
!pip install torch transformers diffusers accelerate torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 28.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [ ]:
import pandas as pd
import base64
import io
from PIL import Image
import torch
from transformers import LlavaForConditionalGeneration, LlavaProcessor

# Load CSV file
csv_path = "../dataset/training/fully_annotated_dataset.csv"  # Change this to your actual file path
df = pd.read_csv(csv_path)

# Initialize LLaVA model and processor
model_name = "llava-hf/llava-1.5-7b-hf"  # You can use another LLaVA variant
processor = LlavaProcessor.from_pretrained(model_name)
model = LlavaForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda" if torch.cuda.is_available() else "cpu")

# Function to decode base64 images
def decode_and_preprocess_base64_image(base64_string):
    try:
        base64_data = base64_string.split(",")[-1]  # Remove prefix like 'data:image/png;base64,'
        image_data = base64.b64decode(base64_data)
        image = Image.open(io.BytesIO(image_data)).convert("RGB")  # Convert to RGB
        image = image.resize((336, 336))  # Resize to model-compatible dimensions
        return image
    except Exception as e:
        return None

# Function to classify emotion
def classify_emotion(image):
    if image is None:
        return "Error: Invalid image"
    inputs = processor(images=image, text="Describe the emotion in the image as Positive, Negative, or Neutral.", return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    output = model.generate(**inputs)
    response = processor.batch_decode(output, skip_special_tokens=True)[0]
    return response

# Process all images in the CSV
emotions = []
for base64_string in df["filename"]:
    try:
        image = decode_and_preprocess_base64_image(base64_string)
        emotion = classify_emotion(image)
    except Exception as e:
        emotion = f"Error: {str(e)}"
    emotions.append(emotion)

# Add results to CSV
df["emotion_img"] = emotions
output_csv_path = "../outputs/llava_baseline.csv"
df.to_csv(output_csv_path, index=False)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 14.12 MiB is free. Process 2349 has 14.72 GiB memory in use. Of the allocated memory 14.60 GiB is allocated by PyTorch, and 1.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

VISION TRANSFORMERS

In [ ]:
!pip install transformers torch torchvision pandas pillow


In [ ]:
import pandas as pd

# Load the two CSV files
df1 = pd.read_csv('../dataset/splits/train_data_img.csv')
df2 = pd.read_csv('../dataset/splits/validation_data_img.csv')

# Merge the data (stack them on top of each other)
merged_df = pd.concat([df1, df2], ignore_index=True)

# Optionally, drop duplicates if needed
# merged_df = merged_df.drop_duplicates()

# Save to a new CSV
merged_df.to_csv('../outputs/merged_file_train_img.csv', index=False)


In [ ]:
import torch
import pandas as pd
import base64
from PIL import Image
from io import BytesIO
from torchvision import transforms
from transformers import ViTForImageClassification
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [ ]:
class EmotionBase64Dataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform
        self.label_map = {
            'positive': 0,
            'Negative': 1,
            'Neutral': 2
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        base64_str = self.data.iloc[index]['filename']
        label = self.label_map[self.data.iloc[index]['post_mood_PNN']]

        # Decode base64 to image
        image_data = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_data)).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


In [ ]:
csv_path = '../outputs/merged_file_train_img.csv'  # Path to your CSV file

dataset = EmotionBase64Dataset(csv_file=csv_path, transform=transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [ ]:
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=3,
    id2label={0: 'positive', 1: 'Negative', 2: 'Neutral'},
    label2id={'positive': 0, 'Negative': 1, 'Neutral': 2},
    ignore_mismatched_sizes=True
)


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
# Training Loop
num_epochs = 7
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%')

# Save Model
torch.save(model.state_dict(), '../models/emotion_vit_base64.pth')

print("Training complete. Model saved as 'emotion_vit_base64.pth'")


Epoch [1/7], Loss: 0.8685, Accuracy: 62.05%
Epoch [2/7], Loss: 0.2497, Accuracy: 95.58%
Epoch [3/7], Loss: 0.0667, Accuracy: 100.00%
Epoch [4/7], Loss: 0.0239, Accuracy: 100.00%
Epoch [5/7], Loss: 0.0141, Accuracy: 100.00%
Epoch [6/7], Loss: 0.0081, Accuracy: 100.00%
Epoch [7/7], Loss: 0.0054, Accuracy: 100.00%
Training complete. Model saved as 'emotion_vit_base64.pth'


In [ ]:
class TestEmotionBase64Dataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        base64_str = self.data.iloc[index]['filename']

        image_data = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_data)).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image


In [ ]:
model.load_state_dict(torch.load('../models/emotion_vit_base64.pth'))
model.eval()


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
inverse_label_map = {0: 'positive', 1: 'Negative', 2: 'Neutral'}


In [ ]:
test_csv_path = '../dataset/splits/test_data_img.csv'  # Your test CSV

# Prepare dataset and loader
test_dataset = TestEmotionBase64Dataset(csv_file=test_csv_path, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Predict labels
predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images).logits
        _, predicted = outputs.max(1)
        predictions.extend(predicted.cpu().numpy())

# Map back to class names
predicted_labels = [inverse_label_map[p] for p in predictions]

# Update CSV with predictions
test_df = pd.read_csv(test_csv_path)
test_df['predicted_post_mood_PNN'] = predicted_labels
test_df.to_csv('../outputs/test_emotions_with_predictions_7.csv', index=False)

print("Predictions saved to 'test_emotions_with_predictions4.csv'")


Predictions saved to 'test_emotions_with_predictions4.csv'


AGUMENTED IS NOT BETTER THAN THE NON AGUMENTED.



---



AGUMENTED:
Metrics per class:

---


Label: negative
  Precision: 0.7674
  Recall: 0.8049
  F1 Score: 0.7857

---


Label: neutral
  Precision: 0.8235
  Recall: 0.6829
  F1 Score: 0.7467

---


Label: positive
  Precision: 0.6522
  Recall: 0.7317
  F1 Score: 0.6897


---
NON AGUMENTED
Metrics per class:

---


Label: negative
  Precision: 0.8857
  Recall: 0.7561
  F1 Score: 0.8158

---


Label: neutral
  Precision: 0.8667
  Recall: 0.6341
  F1 Score: 0.7324

---


Label: positive
  Precision: 0.6552
  Recall: 0.9268
  F1 Score: 0.7677

CLIP - finetuning

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from tqdm import tqdm
import base64
from io import BytesIO


In [ ]:
csv_path = "../dataset/splits/train_data_img.csv"  # Change to your actual CSV file path in Colab
df = pd.read_csv(csv_path)

# Load CLIP Model & Processor
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Dataset Class to Read Images and Text from CSV
class ImageTextDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe
        self.image_base64 = self.df['filename'].tolist()   # This column contains base64 strings
        self.texts = self.df['post_mood_PNN'].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Decode base64 image to PIL Image
        base64_str = self.image_base64[idx]
        image_bytes = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_bytes)).convert("RGB")

        # Process image using CLIP processor
        image = processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        # Process text
        text = processor.tokenizer(self.texts[idx], padding="max_length", truncation=True, return_tensors="pt")["input_ids"].squeeze(0)

        return image, text

# Create DataLoader
dataset = ImageTextDataset(df)
train_dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Optimizer and Loss
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, betas=(0.9, 0.98), eps=1e-6, weight_decay=0.2)
loss_img = nn.CrossEntropyLoss()
loss_txt = nn.CrossEntropyLoss()



In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    epoch_loss = 0.0
    correct_image_to_text = 0
    correct_text_to_image = 0
    total_samples = 0

    pbar = tqdm(train_dataloader, total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in pbar:
        optimizer.zero_grad()

        images, texts = batch
        images = images.to(device)
        texts = texts.to(device)

        # Forward pass
        outputs = model(pixel_values=images, input_ids=texts)
        logits_per_image = outputs.logits_per_image
        logits_per_text = outputs.logits_per_text

        # Ground truth (diagonal matching)
        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Loss Calculation
        loss = (loss_img(logits_per_image, ground_truth) + loss_txt(logits_per_text, ground_truth)) / 2
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Accuracy Calculation
        with torch.no_grad():
            image_to_text_preds = logits_per_image.argmax(dim=1)
            text_to_image_preds = logits_per_text.argmax(dim=1)

            correct_image_to_text += (image_to_text_preds == ground_truth).sum().item()
            correct_text_to_image += (text_to_image_preds == ground_truth).sum().item()
            total_samples += len(images)

        pbar.set_postfix(loss=loss.item())

    # Average loss and accuracy after each epoch
    epoch_loss /= len(train_dataloader)
    img_to_txt_acc = correct_image_to_text / total_samples
    txt_to_img_acc = correct_text_to_image / total_samples
    epoch_accuracy = (img_to_txt_acc + txt_to_img_acc) / 2

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy*100:.2f}%")

torch.save(model.state_dict(), "../models/clip_model.pth")
print("Model saved to ../models/clip_model.pth")


Epoch 1/5: 100%|██████████| 73/73 [00:35<00:00,  2.08it/s, loss=0]


Epoch 1/5 - Loss: 2.0487, Accuracy: 14.82%


Epoch 2/5: 100%|██████████| 73/73 [00:32<00:00,  2.26it/s, loss=0]


Epoch 2/5 - Loss: 2.0511, Accuracy: 14.90%


Epoch 3/5: 100%|██████████| 73/73 [00:32<00:00,  2.27it/s, loss=0]


Epoch 3/5 - Loss: 2.0526, Accuracy: 14.21%


Epoch 4/5: 100%|██████████| 73/73 [00:31<00:00,  2.28it/s, loss=0]


Epoch 4/5 - Loss: 2.0616, Accuracy: 14.38%


Epoch 5/5: 100%|██████████| 73/73 [00:31<00:00,  2.33it/s, loss=0]


Epoch 5/5 - Loss: 2.0492, Accuracy: 13.86%
Model saved to ../models/clip_model.pth


In [ ]:
#test
class TestEmotionBase64Dataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        base64_str = self.data.iloc[index]['filename']

        image_data = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_data)).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image

test_csv_path = '../dataset/splits/test_data_img.csv'
test_dataset = TestEmotionBase64Dataset(csv_file=test_csv_path, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model.load_state_dict(torch.load('../models/clip_emotion_base64.pth'))
model.eval()
predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        outputs = model(pixel_values=images, input_ids=text_inputs.input_ids,
                        attention_mask=text_inputs.attention_mask)

        logits_per_image = outputs.logits_per_image
        predicted_indices = logits_per_image.argmax(dim=1)

        for idx in predicted_indices:
            predictions.append(class_labels[idx.item()])

test_df = pd.read_csv(test_csv_path)
test_df['predicted_post_mood_PNN'] = predictions
test_df.to_csv('../outputs/test_emotions_with_clip_predictions.csv', index=False)

print("Predictions saved to 'test_emotions_with_clip_predictions.csv'")

In [ ]:
#FUSION OF TEXT ROBERTA AND VIT
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, ViTModel
from torchvision import transforms
from PIL import Image
from io import BytesIO
import base64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load fine-tuned RoBERTa with classification head, then extract encoder
text_tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
text_model_cls = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    num_labels=3
)
text_model_cls.load_state_dict(torch.load("../models/roberta_finetuned.pth"))
text_model_cls.to(device).eval()
text_model = text_model_cls.roberta  # Use encoder for CLS embedding

# Load fine-tuned ViT model
vit_model = ViTModel.from_pretrained("google/vit-base-patch16-224")
vit_model.load_state_dict(torch.load("../models/emotion_vit_base64.pth"), strict=False)
vit_model.to(device).eval()

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

In [ ]:
label_map = {'positive': 0, 'Negative': 1, 'Neutral': 2}
inverse_label_map = {v: k for k, v in label_map.items()}

class FusionSentimentDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row["body"])
        image_b64 = row["filename"]
        label = label_map[row["label_post"]]

        encoded_input = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        with torch.no_grad():
            text_emb = text_model(**encoded_input).last_hidden_state[:, 0, :]

        image_data = base64.b64decode(image_b64)
        image = Image.open(BytesIO(image_data)).convert('RGB')
        image_tensor = self.transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            image_emb = vit_model(image_tensor).last_hidden_state[:, 0, :]

        return text_emb.squeeze(0), image_emb.squeeze(0), torch.tensor(label)


In [ ]:
class FusionClassifier(nn.Module):
    def __init__(self, text_dim=768, image_dim=768, hidden_dim=512, num_labels=3):
        super(FusionClassifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(text_dim + image_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, text_emb, image_emb):
        x = torch.cat((text_emb, image_emb), dim=1)
        return self.fc(x)


In [ ]:

def train_fusion_model(train_csv, val_csv, num_epochs=5, batch_size=16):
    train_data = FusionSentimentDataset(train_csv)
    val_data = FusionSentimentDataset(val_csv)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)

    model = FusionClassifier().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for text_emb, image_emb, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(text_emb, image_emb)
            loss = criterion(outputs, labels.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for text_emb, image_emb, labels in val_loader:
                outputs = model(text_emb, image_emb)
                preds = outputs.argmax(dim=1)
                correct += (preds.cpu() == labels).sum().item()
                total += labels.size(0)

        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_loss/len(train_loader):.4f}, Val Acc: {100 * correct/total:.2f}%")

    torch.save(model.state_dict(), "../models/fusion_sentiment_model.pth")
    print("Fusion model saved at ../models/fusion_sentiment_model.pth")


In [ ]:
train_fusion_model(
    train_csv="../dataset/training/train_data_post.csv",
    val_csv="../dataset/validation/validation_data_post.csv",
    num_epochs=4
)

In [ ]:
#testing the FUSION MODEL
fusion_model = FusionClassifier().to(device)
fusion_model.load_state_dict(torch.load("../models/fusion_sentiment_model.pth"))
fusion_model.eval()

In [ ]:
# Define image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load test data
test_df = pd.read_csv("../dataset/testing/test_data_post.csv")
predicted_labels = []

# Run inference
with torch.no_grad():
    for idx, row in test_df.iterrows():
        # Text
        text = str(row['body'])
        inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        text_emb = text_model(**inputs).last_hidden_state[:, 0, :]  # CLS token

        # Image
        base64_str = row['filename']
        image_data = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_data)).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
        image_emb = vit_model(image_tensor).last_hidden_state[:, 0, :]  # CLS token

        # Fusion prediction
        output = fusion_model(text_emb, image_emb)
        pred = torch.argmax(output, dim=1).item()
        predicted_labels.append(inverse_label_map[pred])

# Add predictions to CSV
test_df["predicted_post_sentiment"] = predicted_labels
test_df.to_csv("../models/test_data_post_with_predictions.csv", index=False)

print("Predictions saved to ../models/test_data_post_with_predictions.csv")